# ResNet From Scratch — Colab runner

Runs an experiment from `experiments/*/experiment.yaml` on a Colab GPU and writes the
results back to Google Drive (so a dropped session loses nothing) and, optionally, to the
GitHub repo.

**Before running:** set `Runtime -> Change runtime type -> T4 GPU`, then fill in the two
variables in the next cell.


In [ ]:
# --- settings -------------------------------------------------------------
REPO_URL   = "https://github.com/<you>/resnet-research.git"  # <-- your repo
BRANCH     = "main"
EXPERIMENT = "experiments/01_depth/experiment.yaml"

# everything persistent lives under this Drive folder
DRIVE_DIR  = "/content/drive/MyDrive/resnet-research"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
pathlib.Path(DRIVE_DIR, 'data').mkdir(parents=True, exist_ok=True)
pathlib.Path(DRIVE_DIR, 'results').mkdir(parents=True, exist_ok=True)


In [ ]:
# clone (or update) the repo
import os
if not os.path.isdir('/content/repo'):
    !git clone --branch $BRANCH $REPO_URL /content/repo
%cd /content/repo
!git pull --ff-only

# symlink data/ and results/ to Drive so they survive a session drop
!rm -rf data results
!ln -s $DRIVE_DIR/data data
!ln -s $DRIVE_DIR/results results


In [ ]:
!pip -q install -e . torchinfo

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


In [ ]:
# download CIFAR-10 into Drive once (skipped if already there)
!python scripts/prepare_data.py --root data


In [ ]:
# run the experiment. finished runs (results/runs/<name>/summary.json) are skipped,
# so it is safe to re-run this cell after a disconnect.
!python scripts/run_experiment.py --experiment $EXPERIMENT --device cuda --num-workers 2


In [ ]:
# evaluate every finished run on the held-out test set
import glob, os
for run in sorted(glob.glob('results/runs/*')):
    if os.path.exists(os.path.join(run, 'summary.json')):
        !python scripts/evaluate.py --run $run --device cuda


In [ ]:
# show the figures
from IPython.display import Image, display
import glob
exp_dir = os.path.dirname(EXPERIMENT)
for png in sorted(glob.glob(f'{exp_dir}/figures/*.png')):
    print(png); display(Image(png))


## Commit results back to the repo (optional)

Needs a GitHub personal access token with `repo` scope. Store it as a Colab secret named
`GH_TOKEN` (key icon in the sidebar) — it is never printed.


In [ ]:
from google.colab import userdata
token = userdata.get('GH_TOKEN')

!git config user.email 'szinprazapati1@gmail.com'
!git config user.name 'Szin7'
!git add results/runs experiments/*/figures results/figures 2>/dev/null
!git commit -m 'results: run $EXPERIMENT on Colab T4' || echo 'nothing to commit'

push_url = REPO_URL.replace('https://', f'https://{token}@')
!git push $push_url $BRANCH


If you would rather not use a token, the results are already on Drive under
`resnet-research/results/` — copy that folder into your local clone and commit from there.
